In [23]:
# !pip install twikit

In [24]:
from twikit import Client, TooManyRequests
import pandas as pd
import asyncio
from datetime import datetime

In [25]:
# ---- LOGIN ----
cookies = {
    "auth_token": "7ff2074bbb31051133c6110d82ac9c67d498d07b",
    "ct0": "19f62c63e61204b687fe27c013ea84cdcddc668275a38dfcc39ab35323f14574fe827ed3033bfc71dc9179b6e9a0bd5c91d74f5689e52cca2d888e04531ec53d2671fa70eef802f8f67d142125241a1c"
}

client = Client("en-US")
client.set_cookies(cookies)

QUERY = "(bitcoin OR btc OR $btc) lang:en -is:retweet"

In [26]:
# ---- CLEAN LIVE FETCH FUNCTION ----
async def fetch_btc_tweets(limit=50):
    tweets = None
    data = []
    seen = set()

    while len(data) < limit:
        try:
            # first fetch
            if tweets is None:
                tweets = await client.search_tweet(QUERY, product="Latest")
            else:
                tweets = await tweets.next()

            if not tweets:
                print("No new tweets yet. Waiting...")
                await asyncio.sleep(3)
                continue

            collected = 0
            for t in tweets:
                if len(data) >= limit:
                    break

                if t.id in seen:
                    continue

                seen.add(t.id)
                collected += 1

                data.append({
                    "timestamp": pd.to_datetime(t.created_at),
                    "tweet": t.text
                })

            if collected:
                print(f"Collected {collected} new tweet(s). Total: {len(data)}")
            else:
                print("No new tweets found in this batch. Waiting...")

        except TooManyRequests as e:
            wait = max((datetime.fromtimestamp(e.rate_limit_reset) - datetime.now()).seconds, 20)
            print(f"Rate limit reached. Waiting {int(wait)}s...")
            await asyncio.sleep(wait)
            tweets = None

        except Exception:
            print("Error occurred. Retrying in 3s...")
            await asyncio.sleep(3)
            tweets = None

    return pd.DataFrame(data)


In [27]:
# ---- RUN IN COLAB ----
df = await fetch_btc_tweets(limit=50)


Collected 14 new tweet(s). Total: 14
Collected 16 new tweet(s). Total: 30
Collected 18 new tweet(s). Total: 48
Collected 2 new tweet(s). Total: 50


In [28]:
df['timestamp'] = df['timestamp'].dt.tz_convert('Asia/Kolkata').dt.strftime('%Y-%m-%d %H:%M:%S')

In [31]:
df

,timestamp,tweet
0,2025-12-11 19:22:59,@btc_investpod that was fun! \n\nWe're back a...
1,2025-12-11 19:22:59,@cz_binance Really Dumb to invest in Scam btc ...
2,2025-12-11 19:22:58,🔥 The BIGGЕST #Сryрtо #РUMР #Signаl is here! 🚀...
3,2025-12-11 19:22:57,🔥 The BIGGЕST #Сryрtо #РUMР #Signаl is here! 🚀...
4,2025-12-11 19:22:57,@Bitcoin_Teddy Bitcoin
5,2025-12-11 19:22:56,Imagine 😂\n$BTC https://t.co/AtBfMr2pyj
6,2025-12-11 19:22:56,This is only the beginning of institutional Bi...
7,2025-12-11 19:22:55,@QuintenFrancois Never sell your bitcoin
8,2025-12-11 19:22:55,"@great_martis 87K is key, but not the end. $BT..."
9,2025-12-11 19:22:54,@SurfAI @Jules0712089046 @0x_dingus Just quick...


In [30]:
# from twikit import Client, TooManyRequests
# from datetime import datetime
# import time
# import asyncio

# # ---- COOKIES ----
# cookies = {
#     "auth_token": "7ff2074bbb31051133c6110d82ac9c67d498d07b",
#     "ct0": "19f62c63e61204b687fe27c013ea84cdcddc668275a38dfcc39ab35323f14574fe827ed3033bfc71dc9179b6e9a0bd5c91d74f5689e52cca2d888e04531ec53d2671fa70eef802f8f67d142125241a1c"
# }

# # ---- LOGIN ----
# client = Client("en-US")
# client.set_cookies(cookies)
# print("Logged in!")

# QUERY = "(from:elonmusk) lang:en"

# # ---- LIVE FETCH ----
# async def live_tweets():
#     tweets = None
#     print("Streaming live tweets...\n")

#     while True:
#         try:
#             if tweets is None:
#                 tweets = await client.search_tweet(QUERY, product="Latest")
#             else:
#                 tweets = await tweets.next()

#             if not tweets:
#                 print("No new tweets. Waiting...")
#                 await asyncio.sleep(5)
#                 continue

#             for t in tweets:
#                 print("\n----------------------")
#                 print("User:", t.user.name)
#                 print("Time:", t.created_at)
#                 print("Text:", t.text)
#                 print("----------------------")

#         except TooManyRequests as e:
#             reset = datetime.fromtimestamp(e.rate_limit_reset)
#             wait = (reset - datetime.now()).seconds
#             print(f"Rate limit hit! Waiting {wait}s")
#             time.sleep(wait)

# # ---- RUN (COLAB SAFE) ----
# await live_tweets()
